# NIFTY 50 Incremental Downloader + Parquet + DuckDB

Standalone Google Colab notebook.

Existing CSV:
`/content/drive/MyDrive/quant/data/indices/nifty50/NIFTY50.csv`

Every run:
1. Reads the existing CSV.
2. Finds the latest `Date`.
3. Downloads only data after that date from Yahoo Finance `^NSEI`.
4. Merges and deduplicates.
5. Overwrites `NIFTY50.csv`.
6. Writes/updates `NIFTY50.parquet`.
7. Runs DuckDB sanity checks against the actual CSV and Parquet files.

If the CSV does not exist or is empty, the first download starts at 2024-01-01.


In [ ]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip -q install yfinance pyarrow duckdb


In [ ]:
# ============================================================
# 2. IMPORTS + GOOGLE DRIVE
# ============================================================

from pathlib import Path

import pandas as pd
import numpy as np
import yfinance as yf
import duckdb

from google.colab import drive

drive.mount("/content/drive")

print("Ready.")


In [ ]:
# ============================================================
# 3. CONFIGURATION
# ============================================================

NIFTY_SYMBOL = "^NSEI"

INITIAL_START_DATE = pd.Timestamp("2024-01-01")

NIFTY_DIR = Path(
    "/content/drive/MyDrive/quant/data/indices/nifty50"
)

NIFTY_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

NIFTY_CSV = NIFTY_DIR / "NIFTY50.csv"
NIFTY_PARQUET = NIFTY_DIR / "NIFTY50.parquet"

print("=" * 80)
print("NIFTY 50 INCREMENTAL DOWNLOADER")
print("=" * 80)
print("Source :", NIFTY_SYMBOL)
print("CSV    :", NIFTY_CSV)
print("Parquet:", NIFTY_PARQUET)
print("Initial start:", INITIAL_START_DATE.date())


In [ ]:
# ============================================================
# 4. LOAD EXISTING CSV
# ============================================================

if NIFTY_CSV.exists():

    existing = pd.read_csv(NIFTY_CSV)

    print("Existing NIFTY50.csv found.")
    print("Rows:", f"{len(existing):,}")

else:

    print("NIFTY50.csv does not exist.")

    existing = pd.DataFrame(
        columns=[
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
        ]
    )


In [ ]:
# ============================================================
# 5. NORMALIZE EXISTING CSV
# ============================================================

column_lookup = {
    str(c).strip().lower(): c
    for c in existing.columns
}

required = [
    "date",
    "open",
    "high",
    "low",
    "close",
]

missing = [
    c for c in required
    if c not in column_lookup
]

if missing:
    raise RuntimeError(
        f"NIFTY50.csv missing columns: {missing}. "
        f"Found: {list(existing.columns)}"
    )

existing = existing.rename(
    columns={
        column_lookup["date"]: "Date",
        column_lookup["open"]: "Open",
        column_lookup["high"]: "High",
        column_lookup["low"]: "Low",
        column_lookup["close"]: "Close",
    }
)

existing = existing[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
    ]
].copy()

existing["Date"] = pd.to_datetime(
    existing["Date"],
    errors="coerce",
).dt.date

for c in ["Open", "High", "Low", "Close"]:
    existing[c] = pd.to_numeric(
        existing[c],
        errors="coerce",
    )

existing = existing.dropna(
    subset=[
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
    ]
)

existing = (
    existing
    .drop_duplicates(
        subset=["Date"],
        keep="last",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

print("=" * 80)
print("EXISTING DATA")
print("=" * 80)

if existing.empty:
    print("No usable rows.")
else:
    print("Rows       :", f"{len(existing):,}")
    print("First date :", existing["Date"].min())
    print("Latest date:", existing["Date"].max())


In [ ]:
# ============================================================
# 6. CALCULATE INCREMENTAL RANGE
# ============================================================

if existing.empty:

    download_start = INITIAL_START_DATE

else:

    download_start = (
        pd.Timestamp(existing["Date"].max())
        + pd.Timedelta(days=1)
    )

# yfinance end date is exclusive.
download_end = (
    pd.Timestamp.today().normalize()
    + pd.Timedelta(days=1)
)

print("=" * 80)
print("INCREMENTAL RANGE")
print("=" * 80)

print(
    "Existing latest:",
    (
        existing["Date"].max()
        if not existing.empty
        else "NONE"
    )
)

print(
    "Download start:",
    download_start.date(),
)

print(
    "Download end:",
    (download_end - pd.Timedelta(days=1)).date(),
)

if download_start >= download_end:
    print("Already up to date. No download required.")


In [ ]:
# ============================================================
# 7. DOWNLOAD ONLY MISSING DATA
# ============================================================

if download_start >= download_end:

    new_data = pd.DataFrame(
        columns=[
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
        ]
    )

else:

    raw = yf.download(
        NIFTY_SYMBOL,
        start=download_start.strftime("%Y-%m-%d"),
        end=download_end.strftime("%Y-%m-%d"),
        interval="1d",
        auto_adjust=False,
        progress=True,
    )

    if raw.empty:

        print("Yahoo Finance returned no new rows.")

        new_data = pd.DataFrame(
            columns=[
                "Date",
                "Open",
                "High",
                "Low",
                "Close",
            ]
        )

    else:

        if isinstance(raw.columns, pd.MultiIndex):

            if NIFTY_SYMBOL in raw.columns.get_level_values(-1):

                raw = raw.xs(
                    NIFTY_SYMBOL,
                    axis=1,
                    level=-1,
                )

            else:

                raw.columns = (
                    raw.columns
                    .get_level_values(0)
                )

        raw = raw.reset_index()

        raw.columns = [
            str(c).strip().lower()
            for c in raw.columns
        ]

        if "date" not in raw.columns:

            if "datetime" in raw.columns:

                raw = raw.rename(
                    columns={"datetime": "date"}
                )

            else:

                raise RuntimeError(
                    f"Could not identify Date column: "
                    f"{list(raw.columns)}"
                )

        missing_new = [
            c for c in ["open", "high", "low", "close"]
            if c not in raw.columns
        ]

        if missing_new:
            raise RuntimeError(
                f"Missing OHLC columns: {missing_new}"
            )

        new_data = raw[
            [
                "date",
                "open",
                "high",
                "low",
                "close",
            ]
        ].copy()

        new_data.columns = [
            "Date",
            "Open",
            "High",
            "Low",
            "Close",
        ]

        new_data["Date"] = pd.to_datetime(
            new_data["Date"],
            errors="coerce",
        ).dt.date

        for c in ["Open", "High", "Low", "Close"]:
            new_data[c] = pd.to_numeric(
                new_data[c],
                errors="coerce",
            )

        new_data = new_data.dropna(
            subset=[
                "Date",
                "Open",
                "High",
                "Low",
                "Close",
            ]
        )

        new_data = new_data[
            (
                pd.to_datetime(new_data["Date"])
                >= download_start
            )
            &
            (
                pd.to_datetime(new_data["Date"])
                < download_end
            )
        ]

        new_data = (
            new_data
            .drop_duplicates(
                subset=["Date"],
                keep="last",
            )
            .sort_values("Date")
            .reset_index(drop=True)
        )

print()
print("New rows:", f"{len(new_data):,}")

if not new_data.empty:
    print(
        "New range:",
        new_data["Date"].min(),
        "->",
        new_data["Date"].max(),
    )


In [ ]:
# ============================================================
# 8. MERGE + DEDUPLICATE
# ============================================================

combined = pd.concat(
    [existing, new_data],
    ignore_index=True,
)

combined = (
    combined
    .drop_duplicates(
        subset=["Date"],
        keep="last",
    )
    .sort_values("Date")
    .reset_index(drop=True)
)

combined = combined[
    [
        "Date",
        "Open",
        "High",
        "Low",
        "Close",
    ]
]

print("=" * 80)
print("MERGED DATA")
print("=" * 80)
print("Old rows :", f"{len(existing):,}")
print("New rows :", f"{len(new_data):,}")
print("Final    :", f"{len(combined):,}")
print("Range    :", combined["Date"].min(), "->", combined["Date"].max())


In [ ]:
# ============================================================
# 9. VALIDATE BEFORE WRITING
# ============================================================

if combined.empty:
    raise RuntimeError("Final NIFTY dataset is empty.")

if list(combined.columns) != [
    "Date", "Open", "High", "Low", "Close"
]:
    raise AssertionError("Unexpected final schema.")

if combined["Date"].duplicated().any():
    raise AssertionError("Duplicate dates remain.")

bad_ohlc = combined[
    (combined["High"] < combined["Low"])
    |
    (combined["Open"] > combined["High"])
    |
    (combined["Open"] < combined["Low"])
    |
    (combined["Close"] > combined["High"])
    |
    (combined["Close"] < combined["Low"])
]

if not bad_ohlc.empty:
    raise AssertionError(
        f"Invalid OHLC rows: {len(bad_ohlc)}"
    )

print("Pre-write validation PASSED.")


In [ ]:
# ============================================================
# 10. OVERWRITE CSV SAFELY
# ============================================================

csv_tmp = NIFTY_CSV.with_suffix(".tmp.csv")

combined.to_csv(
    csv_tmp,
    index=False,
)

csv_tmp.replace(NIFTY_CSV)

print("CSV updated:", NIFTY_CSV)
print("Rows:", f"{len(combined):,}")


In [ ]:
# ============================================================
# 11. WRITE CONSOLIDATED PARQUET
# ============================================================

parquet_data = combined.copy()

parquet_data["Date"] = pd.to_datetime(
    parquet_data["Date"]
)

parquet_tmp = NIFTY_PARQUET.with_suffix(
    ".tmp.parquet"
)

parquet_data.to_parquet(
    parquet_tmp,
    index=False,
    engine="pyarrow",
)

parquet_tmp.replace(
    NIFTY_PARQUET
)

print("=" * 80)
print("PARQUET UPDATED")
print("=" * 80)

print("File:", NIFTY_PARQUET)
print("Rows:", f"{len(parquet_data):,}")
print(
    "Latest:",
    parquet_data["Date"].max().date(),
)


In [ ]:
# ============================================================
# 12. DUCKDB SANITY CHECK
# ============================================================

con = duckdb.connect()

try:

    csv_stats = con.execute(
        f'''
        SELECT
            COUNT(*) AS rows,
            MIN(CAST(Date AS DATE)) AS min_date,
            MAX(CAST(Date AS DATE)) AS max_date,
            COUNT(DISTINCT Date) AS distinct_dates
        FROM read_csv_auto(
            '{NIFTY_CSV.as_posix()}'
        )
        '''
    ).fetchone()

    parquet_stats = con.execute(
        f'''
        SELECT
            COUNT(*) AS rows,
            MIN(CAST(Date AS DATE)) AS min_date,
            MAX(CAST(Date AS DATE)) AS max_date,
            COUNT(DISTINCT Date) AS distinct_dates
        FROM read_parquet(
            '{NIFTY_PARQUET.as_posix()}'
        )
        '''
    ).fetchone()

    csv_duplicates = con.execute(
        f'''
        SELECT COUNT(*)
        FROM (
            SELECT Date
            FROM read_csv_auto(
                '{NIFTY_CSV.as_posix()}'
            )
            GROUP BY Date
            HAVING COUNT(*) > 1
        )
        '''
    ).fetchone()[0]

    parquet_duplicates = con.execute(
        f'''
        SELECT COUNT(*)
        FROM (
            SELECT Date
            FROM read_parquet(
                '{NIFTY_PARQUET.as_posix()}'
            )
            GROUP BY Date
            HAVING COUNT(*) > 1
        )
        '''
    ).fetchone()[0]

    # Compare every OHLC row in both directions.
    csv_minus_parquet = con.execute(
        f'''
        SELECT
            CAST(Date AS DATE) AS Date,
            Open, High, Low, Close
        FROM read_csv_auto(
            '{NIFTY_CSV.as_posix()}'
        )
        EXCEPT
        SELECT
            CAST(Date AS DATE) AS Date,
            Open, High, Low, Close
        FROM read_parquet(
            '{NIFTY_PARQUET.as_posix()}'
        )
        '''
    ).fetchall()

    parquet_minus_csv = con.execute(
        f'''
        SELECT
            CAST(Date AS DATE) AS Date,
            Open, High, Low, Close
        FROM read_parquet(
            '{NIFTY_PARQUET.as_posix()}'
        )
        EXCEPT
        SELECT
            CAST(Date AS DATE) AS Date,
            Open, High, Low, Close
        FROM read_csv_auto(
            '{NIFTY_CSV.as_posix()}'
        )
        '''
    ).fetchall()

finally:
    con.close()

print("=" * 80)
print("DUCKDB SANITY CHECK")
print("=" * 80)

print()
print("CSV")
print("Rows          :", f"{csv_stats[0]:,}")
print("Min date      :", csv_stats[1])
print("Max date      :", csv_stats[2])
print("Distinct dates:", f"{csv_stats[3]:,}")
print("Duplicate dates:", csv_duplicates)

print()
print("PARQUET")
print("Rows          :", f"{parquet_stats[0]:,}")
print("Min date      :", parquet_stats[1])
print("Max date      :", parquet_stats[2])
print("Distinct dates:", f"{parquet_stats[3]:,}")
print("Duplicate dates:", parquet_duplicates)

print()
print("CSV -> Parquet differences:", len(csv_minus_parquet))
print("Parquet -> CSV differences:", len(parquet_minus_csv))

if csv_stats[0] != parquet_stats[0]:
    raise AssertionError("CSV and Parquet row counts differ.")

if csv_stats[1] != parquet_stats[1]:
    raise AssertionError("CSV and Parquet minimum dates differ.")

if csv_stats[2] != parquet_stats[2]:
    raise AssertionError("CSV and Parquet maximum dates differ.")

if csv_duplicates != 0:
    raise AssertionError("CSV contains duplicate dates.")

if parquet_duplicates != 0:
    raise AssertionError("Parquet contains duplicate dates.")

if csv_minus_parquet:
    raise AssertionError("CSV contains rows missing from Parquet.")

if parquet_minus_csv:
    raise AssertionError("Parquet contains rows missing from CSV.")

print()
print("DUCKDB SANITY CHECK PASSED")


In [ ]:
# ============================================================
# 13. FINAL VERIFICATION
# ============================================================

verify_csv = pd.read_csv(NIFTY_CSV)

verify_csv["Date"] = pd.to_datetime(
    verify_csv["Date"],
    errors="coerce",
).dt.date

print("=" * 80)
print("FINAL VERIFICATION")
print("=" * 80)

print("CSV rows     :", f"{len(verify_csv):,}")
print("First date   :", verify_csv["Date"].min())
print("Latest date  :", verify_csv["Date"].max())
print("CSV          :", NIFTY_CSV)
print("Parquet      :", NIFTY_PARQUET)

assert len(verify_csv) == len(combined)
assert verify_csv["Date"].duplicated().sum() == 0

print()
print("FINAL VERIFICATION PASSED")

display(verify_csv.tail(10))


In [ ]:
# ============================================================
# 14. FINAL SUMMARY
# ============================================================

print("=" * 80)
print("NIFTY 50 INCREMENTAL UPDATE COMPLETE")
print("=" * 80)

print()
print("Downloaded this run :", f"{len(new_data):,}")
print("Total CSV rows      :", f"{len(combined):,}")
print("Latest date         :", combined["Date"].max())
print()
print("CSV     :", NIFTY_CSV)
print("Parquet :", NIFTY_PARQUET)

print()
print("Next run will start from:")
print(
    pd.Timestamp(combined["Date"].max())
    .date()
    + pd.Timedelta(days=1)
)
